# Experiment 03 — alternative vault weighting

Parents: [01-initial.ipynb](01-initial.ipynb) and [02-backtest-accepted-dollar-cap.ipynb](02-backtest-accepted-dollar-cap.ipynb), using the same `hyper-ai-v11.py` source, guard record, data snapshot and execution framework.

## Question and predeclared methods

Compare four alternatives with the original inverse-variance control under **both** allocation policies: v11's baseline capacity redistribution and experiment 02's original-request accepted-dollar cap (rejected allocation remains cash). Run each combination independently over the short source-reference period **1 January–5 October 2026** and all available vault history **1 March 2023–5 October 2026**, end exclusive: **20 backtests**, all starting with $150,000.

| Method | Definition |
| --- | --- |
| Original inverse variance | Exact source 90-day inverse variance; 0.01% daily volatility floor. Both allocation policies must replicate the earlier notebooks. |
| Priority 1: inverse volatility + equal blend | 50% equal weights plus 50% normalised inverse **regularised** volatility. No fitted blend coefficient. |
| Priority 2: equal weights | Equal requested budgets across the source-selected core. Baseline redistributes within capacity; dollar-cap policy does not. |
| Priority 3: regularised inverse volatility | Inverse volatility from trailing completed daily returns, with variance shrunk towards a common prior. |
| Priority 4: correlation-aware risk budget | Equal requested variance contributions under a regularised covariance matrix. Capacity/execution constraints can break equality afterwards; measure both stages. |

The fixed risk-estimation window is 90 calendar days. The common volatility prior is the median measured volatility among selected names with at least 30 non-flat returns, floored at **0.1% daily**; if none qualify, use that floor. Confidence is `nonflat_count / (nonflat_count + 30)`. Shrink each variance towards the prior variance, then apply the same 0.1% floor. Non-flat returns are a heuristic for information content, **not** proof of fresh NAV observations: genuine stable marks may also be flat. These regularisation choices are research parameters, deliberately different from the source control.

For priority 4, estimate pairwise correlations from completed overlapping returns with at least 30 observations, use zero for unavailable correlations, shrink 50% towards identity and project to a positive-definite correlation matrix. Solve equal risk budgets by coordinate descent on the convex objective `0.5*x.T*Cov*x - sum(log(x))/N`, then normalise. See the [convex risk-budget formulation](https://www.cvxgrp.org/cvx_short_course/docs/exercises/notebooks/13.20.html). No return forecasts or negative weights are used.

The hook replaces weights **after source selection**, retaining ranking, gates, rank hysteresis, two-day cadence, six-slot/33%-concentration/33%-TVL limits, locked holdings, fees, buy/sell bands and cleanup rules. It does not force identical future membership across independently evolving portfolios. No future candles enter risk estimates. The baseline's greedy capacity redistribution remains unchanged, including its sensitivity to requested-weight order.

## Restored-data results — 7 October 2026

All 20 short/full-history executions completed. Source and cap controls reproduce notebooks 01 and 02. Highest observed short-period CAGR is 41.20% for Short — Baseline redistribution — Original inverse variance, with 1.96 Sharpe and -6.40% maximum drawdown.

### Strategy comparison — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fresh engine backtests on the same pinned publication and merged readers, each starting with $150,000. Full-history runs are retained separately in the saved outputs.

| Strategy | Final equity (USD) | CAGR | Sharpe | Maximum drawdown | Trades | Mean cash |
| --- | --- | --- | --- | --- | --- | --- |
| Short — Baseline redistribution — Original inverse variance | \$194,520.81 | 41.20% | 1.96 | -6.40% | 581 | 8.83% |
| Short — Dollar cap — Original inverse variance | \$160,350.69 | 9.31% | 1.31 | -4.22% | 474 | 51.32% |
| Short — Baseline redistribution — P1: 50/50 inverse volatility + equal | \$187,263.59 | 34.28% | 1.49 | -9.11% | 595 | 12.43% |
| Short — Dollar cap — P1: 50/50 inverse volatility + equal | \$177,349.75 | 24.93% | 1.49 | -8.17% | 571 | 23.76% |
| Short — Baseline redistribution — P2: Equal weights | \$188,032.51 | 35.01% | 1.44 | -9.53% | 541 | 14.29% |
| Short — Dollar cap — P2: Equal weights | \$182,873.01 | 30.10% | 1.51 | -9.35% | 473 | 23.47% |
| Short — Baseline redistribution — P3: Regularised inverse volatility | \$188,482.08 | 35.43% | 1.61 | -7.69% | 595 | 10.79% |
| Short — Dollar cap — P3: Regularised inverse volatility | \$169,228.81 | 17.40% | 1.28 | -6.68% | 571 | 25.56% |
| Short — Baseline redistribution — P4: Correlation-aware risk budget | \$188,575.25 | 35.52% | 1.62 | -7.84% | 599 | 10.45% |
| Short — Dollar cap — P4: Correlation-aware risk budget | \$171,296.33 | 19.30% | 1.39 | -6.71% | 575 | 25.39% |

Both periods completed. Historical benchmark differences are separate from mandatory equality between controls on identical inputs. See [the rerun report](restored-data-rerun-2026-10-07.md) for input and clock provenance.


### Restored leader history rerun — 7 October 2026

This execution uses the freshly downloaded, pinned repaired dataset and the merged leader-share and recorded-cap readers. Strategy settings, searches and period boundaries are unchanged. Historical benchmark differences are displayed separately from mandatory source/control equality on the same inputs. Generated results are isolated under `_artifacts-restored-2026-10-07/03`.


In [1]:
import datetime
from copy import deepcopy
import hashlib
import importlib.util
import logging
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display
from dotenv import load_dotenv

from tradingstrategy.client import Client
from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.execution_context import notebook_execution_context
from tradeexecutor.strategy.parameters import StrategyParameters
from tradeexecutor.strategy.reserve_currency import ReserveCurrency
from tradeexecutor.statistics.key_metric import (
    calculate_cagr,
    calculate_max_drawdown,
    calculate_sharpe,
    calculate_sortino,
)
from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode

load_dotenv(override=True)
setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=1000)

STRATEGY_PATH = Path("/Users/moo/code/strategies/strategy/hyper-ai-v11.py")
GUARD_PATH = Path("/Users/moo/code/strategies/deploy/hyper-ai-v10-new-guard-vault-info.json")
os.environ["HYPER_AI_GUARD_CONFIG_FILE"] = str(GUARD_PATH)
assert STRATEGY_PATH.is_file(), STRATEGY_PATH
assert GUARD_PATH.is_file(), GUARD_PATH

spec = importlib.util.spec_from_file_location("hyper_ai_v11", STRATEGY_PATH)
strategy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy)
source_sha256 = hashlib.sha256(STRATEGY_PATH.read_bytes()).hexdigest()
parameters = StrategyParameters.from_class(strategy.Parameters)
client = Client.create_jupyter_client()

display(pd.DataFrame([
    {"Input": "Strategy source", "Value": str(STRATEGY_PATH)},
    {"Input": "Strategy SHA-256", "Value": source_sha256},
    {"Input": "Guard record", "Value": str(GUARD_PATH)},
    {"Input": "Decision cycle", "Value": str(parameters.cycle_duration)},
    {"Input": "Initial equity (USD)", "Value": parameters.initial_cash},
]))

RESEARCH_DIR = Path("/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd")
sys.path.insert(0, str(RESEARCH_DIR))
from accepted_dollar_cap import AuditedAlphaModel, make_decide_trades, calculate_position_profit_curves
from tradeexecutor.analysis.weights import calculate_asset_weights
from tqdm_loggable.auto import tqdm


from weighting_methods import WeightingConfig, WeightingProvider, METHOD_NAMES, make_weighted_decide_trades
import numpy as np

config = WeightingConfig(window=int(parameters.inverse_vol_window))
assert config.window == 90
assert parameters.initial_cash == 150_000
assert parameters.max_assets_in_portfolio == parameters.max_capacity_assets == 6
assert parameters.max_concentration_pct == parameters.per_position_cap_of_pool_pct == .33
ARTIFACT_DIR = RESEARCH_DIR / "_artifacts-restored-2026-10-07/03"
ARTIFACT_DIR.mkdir(parents=True,exist_ok=True)
display(pd.DataFrame([
    {"Input": "Parent notebooks", "Value": "01-initial.ipynb; 02-backtest-accepted-dollar-cap.ipynb"},
    {"Input": "Guard SHA-256", "Value": hashlib.sha256(GUARD_PATH.read_bytes()).hexdigest()},
    {"Input": "Cap harness SHA-256", "Value": hashlib.sha256((RESEARCH_DIR / "accepted_dollar_cap.py").read_bytes()).hexdigest()},
    {"Input": "Weighting harness SHA-256", "Value": hashlib.sha256((RESEARCH_DIR / "weighting_methods.py").read_bytes()).hexdigest()},
]))
display(pd.DataFrame([config.__dict__]))

from score_matrix_visualisation import with_backtest_period

def backtest_period_label(start, end):
    return f"{start:%Y-%m-%d} → {end:%Y-%m-%d} (end exclusive, UTC)"


,Input,Value
0,Strategy source,/Users/moo/code/strategies/strategy/hyper-ai-v...
1,Strategy SHA-256,5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e...
2,Guard record,/Users/moo/code/strategies/deploy/hyper-ai-v10...
3,Decision cycle,CycleDuration.cycle_2d
4,Initial equity (USD),150000


,Input,Value
0,Parent notebooks,01-initial.ipynb; 02-backtest-accepted-dollar-...
1,Guard SHA-256,d6af0e4d018af423e05665eadabd11e46794ddf0e00cc3...
2,Cap harness SHA-256,d198d05f232e9ecaec600f90db106c758f164e3a72203a...
3,Weighting harness SHA-256,d104994e7b38b57ea2b2e4d263753b9c81a157e8242d60...


,window,equal_blend,prior_count,sigma_floor,correlation_shrinkage,correlation_min_periods
0,90,0.5,30,0.001,0.5,30


## Prepare the universe and causal risk histories

Reuse the parent universe and indicators once. The risk provider caches estimates by completed decision date and selected set; each sample ends strictly before the decision.

In [2]:
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline

universe_options = UniverseOptions.from_strategy_parameters_class(
    strategy.Parameters, notebook_execution_context
)
universe_input = CreateTradingUniverseInput(
    execution_context=notebook_execution_context,
    client=client,
    timestamp=None,
    parameters=parameters,
    universe_options=universe_options,
    execution_model=None,
)
print("Loading v11 universe and price history; source run used the Trading Strategy vault dataset.")
import sys
sys.path.insert(0, "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd")
from repaired_data_replay import load_repaired_universe, INDICATOR_DIR, repaired_indicator_storage
strategy_universe = load_repaired_universe(strategy, universe_input)
print(f"Loaded {strategy_universe.get_pair_count()} pairs; calculating indicators with one worker.")
indicator_data = calculate_and_load_indicators_inline(
    indicator_storage_path=INDICATOR_DIR,
    strategy_universe=strategy_universe,
    parameters=parameters,
    create_indicators=strategy.create_indicators,
    execution_context=notebook_execution_context,
    max_workers=1,
)
print(f"Indicator combinations: {len(indicator_data.indicator_combinations)}")

# Exclude benchmark pairs when determining the available vault history.
vault_history = []
for pair in strategy_universe.iterate_pairs():
    if pair.is_vault():
        candles = strategy_universe.data_universe.candles.get_candles_by_pair(pair.internal_id)
        valid = candles.loc[candles["close"].gt(0)]
        if len(valid):
            vault_history.append({
                "Pair id": pair.internal_id,
                "First observation": valid.index.min(),
                "Last observation": valid.index.max(),
            })
vault_history_df = pd.DataFrame(vault_history)
long_start = pd.Timestamp(vault_history_df["First observation"].min()).to_pydatetime()
assert long_start < parameters.backtest_start
# Preserve the source's exclusive end boundary: no incomplete 5 October candle.
periods = {
    "Short (source reference)": (parameters.backtest_start, parameters.backtest_end),
    "Long (all available vault history)": (long_start, parameters.backtest_end),
}
display(pd.DataFrame([
    {"Period": label, "Start (UTC)": start, "End (exclusive, UTC)": end,
     "Calendar days": (end - start).days, "Vaults with history": len(vault_history_df)}
    for label, (start, end) in periods.items()
]))
provider = WeightingProvider(strategy_universe, config)


,Period,Start (UTC),"End (exclusive, UTC)",Calendar days,Vaults with history
0,Short (source reference),2026-01-01,2026-10-05,277,514
1,Long (all available vault history),2023-03-01,2026-10-05,1314,514


## Run the weighting and allocator comparisons

Five fixed weighting methods × two allocation policies × two windows. Each run starts with fresh state and resets mutable fee estimates. Runs are sequential to bound memory. The runner saves the notebook after each completed cell; generated audit files are ignored by Git. Progress bars report decision-level progress, and the run cells estimate remaining time from completed runs.

In [3]:
import time
runs = {}
run_durations = []
TOTAL_RUNS = 20
mode_names = {"parent": "Baseline redistribution", "accepted_dollar_cap": "Dollar cap"}


In [4]:
period_label = 'Short (source reference)'
method = 'inverse_variance'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Baseline redistribution — Original inv...,150000.0,194520.812686,0.298078,0.412004,1.95959,3.429825,-0.064031,139,101,6,581,0.088321


In [5]:
period_label = 'Short (source reference)'
method = 'inverse_variance'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Dollar cap — Original inverse variance,150000.0,160350.685162,0.069653,0.093133,1.309515,1.791249,-0.042229,139,112,6,474,0.513205


In [6]:
period_label = 'Short (source reference)'
method = 'blend'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Baseline redistribution — P1: 50/50 in...,150000.0,187263.589557,0.249649,0.34276,1.485152,2.430447,-0.091056,139,101,6,595,0.124345


In [7]:
period_label = 'Short (source reference)'
method = 'blend'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Dollar cap — P1: 50/50 inverse volatil...,150000.0,177349.754184,0.183272,0.249256,1.489335,2.227464,-0.081668,139,101,6,571,0.237639


In [8]:
period_label = 'Short (source reference)'
method = 'equal'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Baseline redistribution — P2: Equal we...,150000.0,188032.506835,0.25478,0.350056,1.442924,2.314945,-0.095311,139,101,6,541,0.142927


In [9]:
period_label = 'Short (source reference)'
method = 'equal'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Dollar cap — P2: Equal weights,150000.0,182873.008924,0.220148,0.301,1.508986,2.287928,-0.09346,139,101,6,473,0.234676


In [10]:
period_label = 'Short (source reference)'
method = 'regularised_inverse_vol'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Baseline redistribution — P3: Regulari...,150000.0,188482.080577,0.25778,0.354327,1.605928,2.68615,-0.076876,139,101,6,595,0.107897


In [11]:
period_label = 'Short (source reference)'
method = 'regularised_inverse_vol'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Dollar cap — P3: Regularised inverse v...,150000.0,169228.813634,0.128985,0.174027,1.278422,1.86768,-0.066824,139,101,6,571,0.255625


In [12]:
period_label = 'Short (source reference)'
method = 'risk_budget'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Baseline redistribution — P4: Correlat...,150000.0,188575.251794,0.258402,0.355212,1.619706,2.691963,-0.078399,139,101,6,599,0.104517


In [13]:
period_label = 'Short (source reference)'
method = 'risk_budget'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Short — Dollar cap — P4: Correlation-aware ris...,150000.0,171296.32538,0.142771,0.193024,1.386979,2.038859,-0.067125,139,101,6,575,0.253917


In [14]:
period_label = 'Long (all available vault history)'
method = 'inverse_variance'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Baseline redistribution — Original inve...,150000.0,254103.348167,0.694022,0.15768,1.23805,1.893162,-0.079151,658,214,6,1074,0.418125


In [15]:
period_label = 'Long (all available vault history)'
method = 'inverse_variance'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Dollar cap — Original inverse variance,150000.0,161561.412935,0.077076,0.020839,0.354569,0.428455,-0.079151,658,215,6,911,0.705743


In [16]:
period_label = 'Long (all available vault history)'
method = 'blend'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Baseline redistribution — P1: 50/50 inv...,150000.0,242130.357006,0.614202,0.142263,1.066859,1.617586,-0.08872,658,214,6,1128,0.438397


In [17]:
period_label = 'Long (all available vault history)'
method = 'blend'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Dollar cap — P1: 50/50 inverse volatili...,150000.0,214137.006951,0.42758,0.103938,1.011578,1.431414,-0.080081,658,214,6,1119,0.519527


In [18]:
period_label = 'Long (all available vault history)'
method = 'equal'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Baseline redistribution — P2: Equal wei...,150000.0,237129.590342,0.580864,0.13566,0.989661,1.49843,-0.11221,658,214,6,1041,0.462793


In [19]:
period_label = 'Long (all available vault history)'
method = 'equal'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Dollar cap — P2: Equal weights,150000.0,221861.474443,0.479076,0.114858,0.977291,1.399628,-0.092898,658,214,6,910,0.511552


In [20]:
period_label = 'Long (all available vault history)'
method = 'regularised_inverse_vol'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Baseline redistribution — P3: Regularis...,150000.0,247926.539088,0.652844,0.149794,1.139663,1.733267,-0.079151,658,214,6,1099,0.425847


In [21]:
period_label = 'Long (all available vault history)'
method = 'regularised_inverse_vol'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Dollar cap — P3: Regularised inverse vo...,150000.0,199429.927525,0.329533,0.082333,0.942543,1.305471,-0.079151,658,214,6,1177,0.563148


In [22]:
period_label = 'Long (all available vault history)'
method = 'risk_budget'
mode = 'parent'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Baseline redistribution — P4: Correlati...,150000.0,242247.860374,0.614986,0.142417,1.081293,1.626438,-0.079151,658,214,6,1105,0.424776


In [23]:
period_label = 'Long (all available vault history)'
method = 'risk_budget'
mode = 'accepted_dollar_cap'
start, end = periods[period_label]
label = f"{period_label.split()[0]} — {mode_names[mode]} — {METHOD_NAMES[method]}"
run_started = time.monotonic()
remaining = TOTAL_RUNS - len(runs)
estimate = (np.mean(run_durations) * remaining / 60) if run_durations else None
print(f"Run {len(runs)+1}/{TOTAL_RUNS}; estimated remaining: {estimate:.1f} minutes" if estimate is not None else "First run; elapsed timings will estimate remaining time.")
allocation_ledger = []
run_parameters = deepcopy(parameters)
run_parameters["backtest_start"] = start
run_parameters["backtest_end"] = end
# Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
strategy.apply_vault_redemption_capital_fee(
    strategy_universe, float(run_parameters.vault_redemption_capital_fee)
)
print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
result = run_backtest_inline(
    engine_version=strategy.trading_strategy_engine_version,
    client=client,
    decide_trades=make_weighted_decide_trades(strategy, mode, method, allocation_ledger, provider),
    universe=strategy_universe,
    indicator_storage=repaired_indicator_storage(strategy_universe),
    indicator_combinations=indicator_data.indicator_combinations,
    parameters=run_parameters,
    cycle_duration=run_parameters.cycle_duration,
    initial_deposit=run_parameters.initial_cash,
    reserve_currency=ReserveCurrency.usdc,
    trade_routing=run_parameters.routing,
    max_slippage=run_parameters.slippage_tolerance_pct,
    start_at=start,
    end_at=end,
    max_workers=1,
    name=f"Experiment 03 — {label}",
    strategy_logging=False,
)
state = result.state
equity = calculate_equity_curve(state)
# Match the daily-resampled metric convention documented in hyper-ai-v11.py.
daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
trades = list(state.portfolio.get_all_trades())
position_counts_by_cycle = {}
for position_stats in state.stats.positions.values():
    for snapshot in position_stats:
        if snapshot.value > 0:
            position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
metrics = {
    "Initial equity (USD)": float(run_parameters.initial_cash),
    "Final equity (USD)": float(equity.iloc[-1]),
    "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
    "CAGR": float(calculate_cagr(daily_returns)),
    "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
    "Sortino": float(calculate_sortino(daily_returns, periods=365)),
    "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
    "Decision cycles": len(cycle_stamps),
    "Distinct positions": len(positions),
    "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
    "Trades": len(trades),
}
daily_weights = calculate_asset_weights(state).unstack().resample("1D").last().reindex(daily_returns.index).ffill().fillna(0.0)
reserve_symbol = state.portfolio.get_default_reserve_asset()[0].token_symbol
daily_equity = equity.resample("1D").last().reindex(daily_returns.index).ffill()
cash_weight = daily_weights[reserve_symbol] / daily_equity
metrics["Mean cash weight"] = float(cash_weight.mean())
allocation_df = pd.DataFrame(allocation_ledger)
if len(allocation_df):
    allocation_df["Avoided refill (USD)"] = (
        allocation_df["Parent accepted, same state (USD)"] - allocation_df["Accepted allocation (USD)"]
    ).clip(lower=0.0)
    if mode == "accepted_dollar_cap":
        assert (allocation_df["Accepted allocation (USD)"] <= allocation_df["Intended allocation (USD)"] + 1e-6).all()
        assert (allocation_df["Final target (USD)"] <= allocation_df["Accepted allocation (USD)"] + 1e-6).all()
assert metrics["Maximum vaults held"] <= 6
if method != "inverse_variance" and len(allocation_df):
    known = allocation_df["Latest risk observation"].notna()
    assert (pd.to_datetime(allocation_df.loc[known, "Latest risk observation"]) < pd.to_datetime(allocation_df.loc[known, "Date"])).all()
    selected = allocation_df.loc[allocation_df["Risk observations"].notna()]
    assert selected["Original weight"].gt(0).all()
    assert selected.groupby("Date")["Original weight"].sum().sub(1).abs().max() < 1e-8
    if method == "risk_budget":
        assert selected.groupby("Date")["Requested risk fraction"].apply(lambda s: (s - 1/len(s)).abs().max()).max() < 1e-7
runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
               "daily_returns": daily_returns, "metrics": metrics,
               "period": period_label, "mode": mode, "method": method, "allocation_df": allocation_df,
               "cash_weight": cash_weight,
               "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
               "First trade": min((t.opened_at for t in trades), default=None)}
run_durations.append(time.monotonic() - run_started)
run = runs[label]
run["allocation_df"].to_csv(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-allocations.csv", index=False)
run["result"].state.write_json_file(ARTIFACT_DIR / f"{period_label.split()[0].lower()}-{mode}-{method}-state.json")
# Results also remain in saved cell outputs if a later cell fails.
display(pd.DataFrame([{"Run": label, **run["metrics"]}]))


,Run,Initial equity (USD),Final equity (USD),Cumulative return,CAGR,Sharpe,Sortino,Maximum drawdown,Decision cycles,Distinct positions,Maximum vaults held,Trades,Mean cash weight
0,Long — Dollar cap — P4: Correlation-aware risk...,150000.0,198260.515928,0.321737,0.080566,0.911282,1.251698,-0.079151,658,214,6,1189,0.563899


## Side-by-side key metrics

The compact table includes all 20 runs, grouped by period, with the two allocation policies side by side. The full trade-executor QuantStats tables retain the earlier notebooks' daily-resampled clock, including zero off-cycle bins. Each return series is evaluated on its own dates. Native-cycle Sharpe is also reported separately; “Time in Market” in QuantStats is not capital deployment.

In [24]:
summary_rows = []
for label, run in runs.items():
    cycle_returns = calculate_returns(run["equity"])
    trades = list(run["result"].state.portfolio.get_all_trades())
    cost = 0.0
    turnover = 0.0
    for trade in trades:
        if not trade.is_success() or not (trade.is_buy() or trade.is_sell()):
            continue
        turnover += float(trade.executed_reserve or 0.0)
        if trade.is_sell():
            gross = abs(float(trade.executed_quantity or 0.0)) * float(trade.planned_mid_price or 0.0)
            cost += gross * float(trade.other_data.get("backtest_vault_redemption_fee", 0.0))
    run["metrics"]["Cycle-clock Sharpe"] = float(calculate_sharpe(cycle_returns, periods=365/2))
    run["metrics"]["Modelled redemption costs (USD)"] = cost
    run["metrics"]["Gross trade turnover (USD)"] = turnover
    summary_rows.append({"Run": label, "Period": run["period"].split()[0], "Allocator": mode_names[run["mode"]],
                         "Weighting": METHOD_NAMES[run["method"]], **run["metrics"]})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(ARTIFACT_DIR / "summary.csv", index=False)
metrics_to_compare = ["CAGR", "Sharpe", "Maximum drawdown", "Mean cash weight", "Final equity (USD)", "Trades"]
side_by_side = summary_df.pivot(index=["Period", "Weighting"], columns="Allocator", values=metrics_to_compare)
display(side_by_side.style.format("{:,.4f}"))
from tradeexecutor.analysis.advanced_metrics import calculate_advanced_metrics, AdvancedMetricsMode
for period_label in periods:
    tables = []
    for label, run in runs.items():
        if run["period"] != period_label:
            continue
        table = calculate_advanced_metrics(run["daily_returns"], mode=AdvancedMetricsMode.full,
                                          periods_per_year=365, convert_to_daily=False, display=True)
        tables.append(table.rename(columns={"Strategy": f"{mode_names[run['mode']]} — {METHOD_NAMES[run['method']]}"}))
    display(pd.concat(tables, axis=1).rename_axis("Metric"))
display(summary_df[["Period", "Allocator", "Weighting", "Cycle-clock Sharpe", "Modelled redemption costs (USD)", "Gross trade turnover (USD)"]])


,Baseline redistribution — Original inverse variance,Dollar cap — Original inverse variance,Baseline redistribution — P1: 50/50 inverse volatility + equal,Dollar cap — P1: 50/50 inverse volatility + equal,Baseline redistribution — P2: Equal weights,Dollar cap — P2: Equal weights,Baseline redistribution — P3: Regularised inverse volatility,Dollar cap — P3: Regularised inverse volatility,Baseline redistribution — P4: Correlation-aware risk budget,Dollar cap — P4: Correlation-aware risk budget
Metric,,,,,,,,,,
Start Period,2026-01-01,2026-01-01,2026-01-01,2026-01-01,2026-01-01,2026-01-01,2026-01-01,2026-01-01,2026-01-01,2026-01-01
End Period,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04
Risk-Free Rate,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
Time in Market,50.0%,48.0%,50.0%,49.0%,50.0%,49.0%,50.0%,49.0%,50.0%,49.0%
Cumulative Return,29.81%,6.97%,24.96%,18.33%,25.48%,22.01%,25.78%,12.9%,25.84%,14.28%
CAGR﹪,41.2%,9.31%,34.28%,24.93%,35.01%,30.1%,35.43%,17.4%,35.52%,19.3%
Sharpe,1.96,1.31,1.49,1.49,1.44,1.51,1.61,1.28,1.62,1.39
Prob. Sharpe Ratio,96.5%,84.83%,90.85%,90.11%,89.99%,90.39%,92.72%,86.5%,92.85%,88.42%
Smart Sharpe,1.94,1.3,1.48,1.48,1.44,1.5,1.6,1.27,1.61,1.38


,Baseline redistribution — Original inverse variance,Dollar cap — Original inverse variance,Baseline redistribution — P1: 50/50 inverse volatility + equal,Dollar cap — P1: 50/50 inverse volatility + equal,Baseline redistribution — P2: Equal weights,Dollar cap — P2: Equal weights,Baseline redistribution — P3: Regularised inverse volatility,Dollar cap — P3: Regularised inverse volatility,Baseline redistribution — P4: Correlation-aware risk budget,Dollar cap — P4: Correlation-aware risk budget
Metric,,,,,,,,,,
Start Period,2023-02-28,2023-02-28,2023-02-28,2023-02-28,2023-02-28,2023-02-28,2023-02-28,2023-02-28,2023-02-28,2023-02-28
End Period,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04,2026-10-04
Risk-Free Rate,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
Time in Market,31.0%,25.0%,31.0%,32.0%,31.0%,31.0%,31.0%,33.0%,31.0%,32.0%
Cumulative Return,69.4%,7.71%,61.42%,42.76%,58.09%,47.91%,65.28%,32.95%,61.5%,32.17%
CAGR﹪,15.77%,2.08%,14.23%,10.39%,13.57%,11.49%,14.98%,8.23%,14.24%,8.06%
Sharpe,1.24,0.35,1.07,1.01,0.99,0.98,1.14,0.94,1.08,0.91
Prob. Sharpe Ratio,98.89%,72.9%,97.69%,96.42%,96.83%,96.2%,98.31%,94.85%,97.79%,94.32%
Smart Sharpe,1.23,0.35,1.06,1.01,0.99,0.97,1.14,0.94,1.08,0.91


,Period,Allocator,Weighting,Cycle-clock Sharpe,Modelled redemption costs (USD),Gross trade turnover (USD)
0,Short,Baseline redistribution,Original inverse variance,1.962856,13078.320667,7.024828e+06
1,Short,Dollar cap,Original inverse variance,1.307869,5950.861878,4.763317e+06
2,Short,Baseline redistribution,P1: 50/50 inverse volatility + equal,1.484285,11306.717749,5.690189e+06
3,Short,Dollar cap,P1: 50/50 inverse volatility + equal,1.488491,8591.601290,4.681722e+06
4,Short,Baseline redistribution,P2: Equal weights,1.441837,10476.646305,5.135075e+06
5,Short,Dollar cap,P2: Equal weights,1.508253,8729.300906,4.235717e+06
6,Short,Baseline redistribution,P3: Regularised inverse volatility,1.605814,11965.496677,6.236602e+06
7,Short,Dollar cap,P3: Regularised inverse volatility,1.276674,8205.436548,5.009697e+06
8,Short,Baseline redistribution,P4: Correlation-aware risk budget,1.619690,11876.804856,6.331103e+06
9,Short,Dollar cap,P4: Correlation-aware risk budget,1.385633,8386.722081,5.041056e+06


## Equity curve comparisons

Each chart compares the five methods within one period and allocation policy.

In [25]:
for period_label in periods:
    for mode in mode_names:
        frames = []
        for label, run in runs.items():
            if run["period"] == period_label and run["mode"] == mode:
                frame = run["equity"].rename("Portfolio equity (USD)").rename_axis("Date").reset_index()
                frame["Weighting"] = METHOD_NAMES[run["method"]]
                frames.append(frame)
        fig = px.line(pd.concat(frames), x="Date", y="Portfolio equity (USD)", color="Weighting",
                      title=f"Experiment 03 — {period_label} — {mode_names[mode]}")
        with_backtest_period(fig, backtest_period_label(*periods[period_label]))
        fig.show(width=1500, height=750)


## Quarterly profit heatmaps

Use the existing trade-executor returns heatmap with calendar-quarter aggregation. Each cell shows compounded portfolio profit/loss in percent, including changes in the value of open positions. The calculation uses the same daily returns as the metrics table.

Boundary quarters contain only observed dates: long-period 2023 Q1 starts with the cash-only 28 February cycle, and 2026 Q4 covers 1–4 October in all runs. These are partial quarters. Blank cells are outside the run's window; 0% means the observed portfolio return was zero.

In [26]:
from matplotlib import pyplot as plt
from tradeexecutor.visual.equity_curve import visualise_returns_over_time

for label, run in runs.items():
    fig = visualise_returns_over_time(run["daily_returns"], period="quarter")
    fig.axes[0].set_title(f"Quarterly portfolio profit/loss — {label}" + "\nBacktest: " + backtest_period_label(run["parameters"].backtest_start, run["parameters"].backtest_end))
    display(fig)
    plt.close(fig)

## Absolute asset allocation

Each stacked chart shows the USD value of every vault holding and reserve cash. Values are absolute, rather than normalised to 100%.

In [27]:
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_asset

for label, run in runs.items():
    result = run["result"]
    run_parameters = run["parameters"]
    charts = strategy.create_charts(
        timestamp=None,
        parameters=run_parameters,
        strategy_universe=result.strategy_universe,
        execution_context=notebook_execution_context,
    )
    chart_renderer = ChartBacktestRenderingSetup(
        registry=charts,
        strategy_input_indicators=result.indicators,
        state=result.state,
        backtest_start_at=run_parameters.backtest_start,
        backtest_end_at=run_parameters.backtest_end,
    )
    asset_weights_fig = chart_renderer.render(equity_curve_by_asset)
    asset_weights_fig.update_layout(title=f"Absolute asset allocation (USD) — {label}", width=1500, height=900)
    # Keep every vault in the static legend visible across the longer history.
    chart_height = max(900, 170 + 20 * len(asset_weights_fig.data))
    with_backtest_period(asset_weights_fig, backtest_period_label(run_parameters.backtest_start, run_parameters.backtest_end))
    asset_weights_fig.show(width=1500, height=chart_height)

## Replication and allocation diagnostics

Both original inverse-variance baseline runs must reproduce experiment 01, and both original inverse-variance cap runs must reproduce experiment 02. Check actual cap targets and causality in every run, then report requested concentration, capital rejection, cleanup interactions and pre-/post-cap risk contributions. Selected membership is unchanged by the weighting hook on the same pre-trade state; later holdings and membership can diverge between strategies.

In [28]:
references = {
    "Short (source reference)": {"Final equity (USD)": 194_929.0, "Cumulative return": 0.3008,
        "CAGR": 0.4159, "Sharpe": 2.00, "Sortino": 3.56, "Maximum drawdown": -0.0640,
        "Decision cycles": 139, "Distinct positions": 101, "Maximum vaults held": 6, "Trades": 582},
    "Long (all available vault history)": {"Final equity (USD)": 254_510.70, "Cumulative return": 0.6967,
        "CAGR": 0.1582, "Sharpe": 1.25, "Sortino": 1.91, "Maximum drawdown": -0.0792,
        "Decision cycles": 658, "Distinct positions": 214, "Maximum vaults held": 6, "Trades": 1074},
}
tolerances = {"Final equity (USD)": 2.0, "Cumulative return": 0.00015, "CAGR": 0.00015,
              "Sharpe": 0.015, "Sortino": 0.015, "Maximum drawdown": 0.00015,
              "Decision cycles": 0, "Distinct positions": 0, "Maximum vaults held": 0, "Trades": 0}
rows = []
for label, run in runs.items():
    if run["mode"] != "parent" or run["method"] != "inverse_variance":
        continue
    for metric, expected in references[run["period"]].items():
        actual = run["metrics"][metric]
        rows.append({"Run": label, "Metric": metric, "Reference": expected, "Rerun": actual,
                     "Match": abs(actual - expected) <= tolerances[metric]})
replication_df = pd.DataFrame(rows)
display(replication_df)
# Historical benchmark comparison; repaired-input parent controls are checked below.

cap_references = {
    "Short": {"Final equity (USD)": 160_369.21, "CAGR": .0933, "Sharpe": 1.31, "Maximum drawdown": -.0422, "Mean cash weight": .5139},
    "Long": {"Final equity (USD)": 161_658.47, "CAGR": .0210, "Sharpe": .36, "Maximum drawdown": -.0792, "Mean cash weight": .7057},
}
for label, run in runs.items():
    if run["mode"] != "accepted_dollar_cap" or run["method"] != "inverse_variance":
        continue
    for metric, expected in cap_references[run["period"].split()[0]].items():
        tolerance = 2 if metric == "Final equity (USD)" else .015 if metric == "Sharpe" else .0002
        actual = run["metrics"][metric]
        rows.append({"Run": label, "Metric": metric, "Reference": expected, "Rerun": actual, "Match": abs(actual - expected) <= tolerance})
replication_df = pd.DataFrame(rows)
display(replication_df)
# Historical cap benchmark comparison; repaired-input cap controls are checked below.
audit_rows = []
for label, run in runs.items():
    ledger = run["allocation_df"]
    positive = ledger[ledger["Raw weight"].gt(0)]
    per_cycle = positive.groupby("Date").agg(budget=("Deployable budget (USD)", "first"),
                                            target=("Final target (USD)", "sum"),
                                            largest_request=("Original weight", "max"))
    row = {"Run": label, "Mean target / fresh budget": (per_cycle.target / per_cycle.budget.replace(0, np.nan)).mean(),
           "Mean largest requested share": per_cycle.largest_request.mean(),
           "Fraction of sized cycles with >90% request": per_cycle.largest_request.gt(.9).mean(),
           "Targets below basket cleanup": int((ledger["Final target (USD)"].gt(0) & ledger["Normalised accepted weight"].lt(parameters.min_portfolio_weight_pct)).sum())}
    if run["method"] == "risk_budget":
        sized = ledger[ledger["Requested risk fraction"].notna()].copy()
        count = sized.groupby("Date")["Pair id"].transform("count")
        row["Max requested risk-budget error"] = (sized["Requested risk fraction"] - 1/count).abs().max()
        row["Mean absolute accepted risk-budget error"] = (sized["Accepted risk fraction"] - 1/count).abs().mean()
    audit_rows.append(row)
audit_df = pd.DataFrame(audit_rows)
display(audit_df)
audit_df.to_csv(ARTIFACT_DIR / "allocation-summary.csv", index=False)

from repaired_data_replay import RUN_ROOT, save_metrics
import json
save_metrics(runs,RUN_ROOT/"03")
fresh_parent=json.loads((RUN_ROOT/"01/metrics.json").read_text())
for run in runs.values():
    if run["mode"]=="parent" and run["method"]=="inverse_variance":
        for metric in ["Final equity (USD)","CAGR","Sharpe","Maximum drawdown","Trades"]:
            assert abs(run["metrics"][metric]-fresh_parent[run["period"]][metric])<1e-6, (run["period"],metric)
fresh_cap=json.loads((RUN_ROOT/"02/metrics.json").read_text())
for run in runs.values():
    if run["mode"]=="accepted_dollar_cap" and run["method"]=="inverse_variance":
        matching=next(m for label,m in fresh_cap.items() if label.startswith(run["period"].split()[0]) and "cap" in label.lower())
        for metric in ["Final equity (USD)","CAGR","Sharpe","Maximum drawdown","Trades"]:
            assert abs(run["metrics"][metric]-matching[metric])<1e-6, (run["period"],metric)


,Run,Metric,Reference,Rerun,Match
0,Short — Baseline redistribution — Original inv...,Final equity (USD),194929.0000,194520.812686,False
1,Short — Baseline redistribution — Original inv...,Cumulative return,0.3008,0.298078,False
2,Short — Baseline redistribution — Original inv...,CAGR,0.4159,0.412004,False
3,Short — Baseline redistribution — Original inv...,Sharpe,2.0000,1.959590,False
4,Short — Baseline redistribution — Original inv...,Sortino,3.5600,3.429825,False
5,Short — Baseline redistribution — Original inv...,Maximum drawdown,-0.0640,-0.064031,True
6,Short — Baseline redistribution — Original inv...,Decision cycles,139.0000,139.000000,True
7,Short — Baseline redistribution — Original inv...,Distinct positions,101.0000,101.000000,True
8,Short — Baseline redistribution — Original inv...,Maximum vaults held,6.0000,6.000000,True
9,Short — Baseline redistribution — Original inv...,Trades,582.0000,581.000000,False


,Run,Metric,Reference,Rerun,Match
0,Short — Baseline redistribution — Original inv...,Final equity (USD),194929.0000,194520.812686,False
1,Short — Baseline redistribution — Original inv...,Cumulative return,0.3008,0.298078,False
2,Short — Baseline redistribution — Original inv...,CAGR,0.4159,0.412004,False
3,Short — Baseline redistribution — Original inv...,Sharpe,2.0000,1.959590,False
4,Short — Baseline redistribution — Original inv...,Sortino,3.5600,3.429825,False
5,Short — Baseline redistribution — Original inv...,Maximum drawdown,-0.0640,-0.064031,True
6,Short — Baseline redistribution — Original inv...,Decision cycles,139.0000,139.000000,True
7,Short — Baseline redistribution — Original inv...,Distinct positions,101.0000,101.000000,True
8,Short — Baseline redistribution — Original inv...,Maximum vaults held,6.0000,6.000000,True
9,Short — Baseline redistribution — Original inv...,Trades,582.0000,581.000000,False


,Run,Mean target / fresh budget,Mean largest requested share,Fraction of sized cycles with >90% request,Targets below basket cleanup,Max requested risk-budget error,Mean absolute accepted risk-budget error
0,Short — Baseline redistribution — Original inv...,0.925352,0.613962,0.100719,0,NaN,NaN
1,Short — Dollar cap — Original inverse variance,0.492929,0.624373,0.115108,60,NaN,NaN
2,Short — Baseline redistribution — P1: 50/50 in...,0.889140,0.222978,0.000000,0,NaN,NaN
3,Short — Dollar cap — P1: 50/50 inverse volatil...,0.773701,0.222978,0.000000,0,NaN,NaN
4,Short — Baseline redistribution — P2: Equal we...,0.869709,0.166667,0.000000,0,NaN,NaN
5,Short — Dollar cap — P2: Equal weights,0.775652,0.166667,0.000000,0,NaN,NaN
6,Short — Baseline redistribution — P3: Regulari...,0.905969,0.279290,0.000000,0,NaN,NaN
7,Short — Dollar cap — P3: Regularised inverse v...,0.755173,0.279290,0.000000,0,NaN,NaN
8,Short — Baseline redistribution — P4: Correlat...,0.909292,0.276913,0.000000,0,9.868843e-09,0.117580
9,Short — Dollar cap — P4: Correlation-aware ris...,0.756952,0.276913,0.000000,0,9.868843e-09,0.073583


## Robustness analysis

Reconstruct each position's PnL from successful USDC trade flows and marked holdings, including final realised profit for closed positions. Reconcile the sum to equity minus the initial deposit, then group by vault address. Show the top vaults and positions and attribute the three strongest portfolio days.

Use the repository's `curator.py` classifications and quarantine windows to annotate influential vaults. Binance BTC and ETH opening prices provide broad-market context over the matching prior-snapshot-to-current-snapshot UTC interval. With two-day decisions, a return booked on one date can cover more than one calendar day; compare that interval rather than a single market day. The best-day-zero and leading-vault-PnL removal figures are fixed-path sensitivities, not executable alternative strategies; they do not establish out-of-sample performance. Cash exposure and the two overlapping windows cannot by themselves separate reduced market exposure from better sizing.

In [29]:
from tradingstrategy.binance.price import fetch_binance_price
from tradeexecutor.curator.curator import EXCLUDED_VAULTS, get_hyperliquid_concentration_tweak, is_quarantined

# Incrementally cached Binance reference series, used only for best-day context.
market_open_prices = {}
for symbol in ("BTCUSDT", "ETHUSDT"):
    prices = fetch_binance_price(symbol=symbol, start=long_start - datetime.timedelta(days=7))
    market_open_prices[symbol] = prices["open"]

curated_quality = get_hyperliquid_concentration_tweak()
robustness_rows = []
contributor_tables = []
jump_tables = []
for label, run in runs.items():
    state = run["result"].state
    daily = run["daily_returns"]
    daily_equity = run["equity"].resample("1D").last().reindex(daily.index).ffill()
    curves, position_metadata = calculate_position_profit_curves(state, daily.index)
    portfolio_pnl = daily_equity - float(run["parameters"].initial_cash)
    residual = curves.sum(axis=1) - portfolio_pnl
    assert residual.abs().max() < 0.01, f"Position attribution does not reconcile: {label}, {residual.abs().max()}"
    vault_curves = {}
    for address, group in position_metadata.groupby("Address"):
        vault_curves[address] = curves[group["Position id"].tolist()].sum(axis=1)
    vault_curves = pd.DataFrame(vault_curves)
    final_profits = vault_curves.iloc[-1].sort_values(ascending=False)
    leader = final_profits.index[0]
    leader_profit = float(final_profits.iloc[0])
    positive_profit_total = final_profits.clip(lower=0.0).sum()
    best_day = daily.idxmax()
    without_best_day = daily.copy()
    without_best_day.loc[best_day] = 0.0
    # Fixed-ledger attribution sensitivity: retain the original trades and sizing.
    without_leader_equity = daily_equity - vault_curves[leader]
    without_leader_returns = calculate_returns(without_leader_equity)
    robustness_rows.append({
        "Run": label, "Best day": best_day, "Best day return": float(daily.loc[best_day]),
        "Return kurtosis (excess)": float(daily.kurt()),
        "CAGR with best day set to zero": float(calculate_cagr(without_best_day)),
        "Sharpe with best day set to zero": float(calculate_sharpe(without_best_day, periods=365)),
        "Leading vault": position_metadata.loc[position_metadata.Address.eq(leader), "Vault"].iloc[0],
        "Leading vault profit (USD)": leader_profit,
        "Leading vault share of positive vault profits": float(leader_profit / positive_profit_total),
        "CAGR without leader PnL (fixed ledger)": float(calculate_cagr(without_leader_returns)),
        "Sharpe without leader PnL (fixed ledger)": float(calculate_sharpe(without_leader_returns, periods=365)),
        "Max attribution residual (USD)": float(residual.abs().max()),
    })
    names = position_metadata.groupby("Address").Vault.first()
    contribution = pd.DataFrame({"Address": final_profits.index, "Profit (USD)": final_profits.values})
    contribution["Vault"] = contribution.Address.map(names)
    contribution["Run"] = label
    contribution["Curated quality"] = contribution.Address.map(lambda address: getattr(curated_quality.get(address), "value", "unrated"))
    contribution["Curator excluded"] = contribution.Address.isin(EXCLUDED_VAULTS)
    contributor_tables.append(contribution.head(5))
    position_metadata["Run"] = label
    run["position_metadata"] = position_metadata
    run["vault_profit_curves"] = vault_curves
    daily_pnl = vault_curves.diff().fillna(vault_curves.iloc[0])
    top_days = daily.nlargest(3).index
    for day in top_days:
        gains = daily_pnl.loc[day].sort_values(ascending=False)
        positive = gains.clip(lower=0.0).sum()
        top_address = gains.index[0]
        snapshot_dates = pd.DatetimeIndex(run["equity"].index).normalize().unique()
        previous_snapshot = snapshot_dates[snapshot_dates < day][-1]
        previous_equity = float(daily_equity.loc[previous_snapshot])
        market_interval_returns = {
            symbol: float(series.loc[day] / series.loc[previous_snapshot] - 1.0)
            for symbol, series in market_open_prices.items()
        }
        jump_tables.append({
            "Run": label, "Date": day, "Portfolio return": float(daily.loc[day]),
            "Largest contributing vault": names[top_address],
            "Vault PnL that day (USD)": float(gains.iloc[0]),
            "Vault contribution / prior equity": float(gains.iloc[0] / previous_equity),
            "Share of positive vault PnL": float(gains.iloc[0] / positive) if positive else 0.0,
            "Previous snapshot": previous_snapshot,
            "BTC over snapshot interval": market_interval_returns["BTCUSDT"],
            "ETH over snapshot interval": market_interval_returns["ETHUSDT"],
            "Vault quarantined that day": is_quarantined(top_address, day.to_pydatetime()),
        })
robustness_df = pd.DataFrame(robustness_rows)
contribution_df = pd.concat(contributor_tables, ignore_index=True)
jumps_df = pd.DataFrame(jump_tables)
display(robustness_df)
display(contribution_df)
display(jumps_df)

position_tables = []
for label, run in runs.items():
    table = run["position_metadata"].sort_values("Final profit (USD)", ascending=False).head(3)
    position_tables.append(table)
display(pd.concat(position_tables, ignore_index=True))
robustness_df.to_csv(ARTIFACT_DIR / "robustness.csv", index=False)
contribution_df.to_csv(ARTIFACT_DIR / "contributors.csv", index=False)
jumps_df.to_csv(ARTIFACT_DIR / "jumps.csv", index=False)
pd.concat(position_tables, ignore_index=True).to_csv(ARTIFACT_DIR / "top-positions.csv", index=False)


,Run,Best day,Best day return,Return kurtosis (excess),CAGR with best day set to zero,Sharpe with best day set to zero,Leading vault,Leading vault profit (USD),Leading vault share of positive vault profits,CAGR without leader PnL (fixed ledger),Sharpe without leader PnL (fixed ledger),Max attribution residual (USD)
0,Short — Baseline redistribution — Original inv...,2026-06-26,0.068871,13.670515,0.292955,1.620008,Realist Capital,24196.229340,0.447703,0.184526,1.097709,3.637979e-11
1,Short — Dollar cap — Original inverse variance,2026-02-28,0.018699,39.344446,0.066675,1.003495,pmalt,4257.495237,0.293620,0.054915,0.822736,2.910383e-11
2,Short — Baseline redistribution — P1: 50/50 in...,2026-06-26,0.067769,8.632958,0.231227,1.143276,Realist Capital,18730.843943,0.387901,0.168075,0.944680,5.093170e-11
3,Short — Dollar cap — P1: 50/50 inverse volatil...,2026-06-26,0.036734,5.956085,0.191055,1.225043,Realist Capital,11042.826211,0.300184,0.147435,1.001316,4.183676e-11
4,Short — Baseline redistribution — P2: Equal we...,2026-06-26,0.064828,7.104387,0.242441,1.124898,Realist Capital,19683.555516,0.395753,0.166390,0.899127,4.365575e-11
5,Short — Dollar cap — P2: Equal weights,2026-06-26,0.040241,6.551805,0.234862,1.261198,Realist Capital,12706.126742,0.291354,0.182818,1.058003,3.637979e-11
6,Short — Baseline redistribution — P3: Regulari...,2026-06-26,0.068652,10.708749,0.240476,1.253890,Realist Capital,21604.432384,0.441296,0.152928,0.896131,2.910383e-11
7,Short — Dollar cap — P3: Regularised inverse v...,2026-06-26,0.033252,6.647901,0.124322,0.987498,Realist Capital,9643.377157,0.351454,0.086376,0.722149,3.274181e-11
8,Short — Baseline redistribution — P4: Correlat...,2026-06-26,0.068438,10.723529,0.241617,1.267708,Realist Capital,21409.626763,0.434192,0.155560,0.920696,5.820766e-11
9,Short — Dollar cap — P4: Correlation-aware ris...,2026-06-26,0.033793,6.439192,0.141724,1.099200,Realist Capital,10439.283963,0.357658,0.097831,0.815877,3.637979e-11


,Address,Profit (USD),Vault,Run,Curated quality,Curator excluded
0,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,24196.229340,Realist Capital,Short — Baseline redistribution — Original inv...,unrated,False
1,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,10689.272691,DOEZOE,Short — Baseline redistribution — Original inv...,unrated,False
2,0x4dec0a851849056e259128464ef28ce78afa27f6,6623.691469,pmalt,Short — Baseline redistribution — Original inv...,high,False
3,0xba939edf38c0ae0cc689c98b492e0535f43e4550,3624.343404,22Cap,Short — Baseline redistribution — Original inv...,unrated,False
4,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2286.660570,Citadel,Short — Baseline redistribution — Original inv...,unrated,False
5,0x4dec0a851849056e259128464ef28ce78afa27f6,4257.495237,pmalt,Short — Dollar cap — Original inverse variance,high,False
6,0xba939edf38c0ae0cc689c98b492e0535f43e4550,2967.269752,22Cap,Short — Dollar cap — Original inverse variance,unrated,False
7,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,1561.554794,Realist Capital,Short — Dollar cap — Original inverse variance,unrated,False
8,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,1428.979487,Citadel,Short — Dollar cap — Original inverse variance,unrated,False
9,0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,1402.659287,Goon Edging,Short — Dollar cap — Original inverse variance,unrated,False


,Run,Date,Portfolio return,Largest contributing vault,Vault PnL that day (USD),Vault contribution / prior equity,Share of positive vault PnL,Previous snapshot,BTC over snapshot interval,ETH over snapshot interval,Vault quarantined that day
0,Short — Baseline redistribution — Original inv...,2026-06-26,0.068871,Realist Capital,6929.951861,0.041740,0.583981,2026-06-24,-0.046863,-0.059545,False
1,Short — Baseline redistribution — Original inv...,2026-06-24,0.041142,Realist Capital,8968.757978,0.056242,1.000000,2026-06-22,-0.009120,-0.023322,False
2,Short — Baseline redistribution — Original inv...,2026-07-08,0.036604,Realist Capital,6207.538547,0.033220,0.870987,2026-07-06,-0.004493,-0.007952,False
3,Short — Dollar cap — Original inverse variance,2026-02-28,0.018699,pmalt,1965.465770,0.012819,0.685539,2026-02-26,-0.031122,-0.062037,False
4,Short — Dollar cap — Original inverse variance,2026-06-26,0.014795,Goon Edging,1983.607541,0.012824,0.738046,2026-06-24,-0.046863,-0.059545,False
5,Short — Dollar cap — Original inverse variance,2026-08-19,0.014300,Citadel,1915.486236,0.012101,0.812906,2026-08-17,0.029021,0.022303,False
6,Short — Baseline redistribution — P1: 50/50 in...,2026-06-26,0.067769,Realist Capital,6811.638859,0.041404,0.588159,2026-06-24,-0.046863,-0.059545,False
7,Short — Baseline redistribution — P1: 50/50 in...,2026-06-24,0.047143,Realist Capital,9506.896668,0.060512,1.000000,2026-06-22,-0.009120,-0.023322,False
8,Short — Baseline redistribution — P1: 50/50 in...,2026-07-08,0.036320,Realist Capital,6144.528988,0.033212,0.876944,2026-07-06,-0.004493,-0.007952,False
9,Short — Dollar cap — P1: 50/50 inverse volatil...,2026-06-26,0.036734,Goon Edging,3421.933699,0.021207,0.538073,2026-06-24,-0.046863,-0.059545,False


,Position id,Vault,Address,Opened,Closed,Final profit (USD),Run
0,64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,23284.926869,Short — Baseline redistribution — Original inv...
1,98,DOEZOE,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,2026-09-18,NaT,11815.190940,Short — Baseline redistribution — Original inv...
2,7,pmalt,0x4dec0a851849056e259128464ef28ce78afa27f6,2026-01-05,2026-05-21,8291.949737,Short — Baseline redistribution — Original inv...
3,88,Citadel,0xda51323fe9800c8365646ad5c7ade0dd17fdc167,2026-08-05,2026-08-27,3012.417960,Short — Dollar cap — Original inverse variance
4,19,pmalt,0x4dec0a851849056e259128464ef28ce78afa27f6,2026-01-25,2026-04-15,2424.056448,Short — Dollar cap — Original inverse variance
5,92,AceVault Hyper01,0x9e02aca9865e1859bb7865f6f64801e804a173df,2026-08-17,2026-08-29,2136.295287,Short — Dollar cap — Original inverse variance
6,64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,17241.023965,Short — Baseline redistribution — P1: 50/50 in...
7,98,DOEZOE,0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,2026-09-18,NaT,11815.190940,Short — Baseline redistribution — P1: 50/50 in...
8,23,Hyperrr,0xca230e816bdb34a46960c2f978a30a563d1ae9e0,2026-01-31,2026-04-21,9056.327081,Short — Baseline redistribution — P1: 50/50 in...
9,64,Realist Capital,0x77fee2df7bad4f1db93052fa82bf78eaab771a16,2026-06-20,2026-08-21,9151.313979,Short — Dollar cap — P1: 50/50 inverse volatil...


## Method versus control contribution attribution

For every period and allocation policy, compare each alternative with the original inverse-variance control. Reconcile the sum of vault PnL differences to the final equity difference, report leading positive and negative differences, and measure how much of a positive gain comes from the largest vault. This includes changes in holdings and fees; it is retrospective attribution, not proof those gains were predictable.

In [30]:
difference_tables = []
for label, run in runs.items():
    if run["method"] == "inverse_variance":
        continue
    control = next(r for r in runs.values() if r["method"] == "inverse_variance" and r["mode"] == run["mode"] and r["period"] == run["period"])
    table = pd.DataFrame({"Alternative PnL (USD)": run["vault_profit_curves"].iloc[-1],
                          "Control PnL (USD)": control["vault_profit_curves"].iloc[-1]}).fillna(0)
    names = pd.concat([run["position_metadata"], control["position_metadata"]]).groupby("Address").Vault.first()
    table["Vault"] = table.index.map(names)
    table["Alternative minus control (USD)"] = table["Alternative PnL (USD)"] - table["Control PnL (USD)"]
    table["Run"] = label
    gap = run["equity"].iloc[-1] - control["equity"].iloc[-1]
    assert abs(table["Alternative minus control (USD)"].sum() - gap) < .01
    display(table.sort_values("Alternative minus control (USD)", ascending=False).head(3))
    display(table.sort_values("Alternative minus control (USD)").head(3))
    difference_tables.append(table.reset_index(names="Address"))
differences_df = pd.concat(difference_tables, ignore_index=True)
differences_df.to_csv(ARTIFACT_DIR / "control-contribution-differences.csv", index=False)


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,3594.973987,305.942864,Hyperrr,3289.031123,Short — Baseline redistribution — P1: 50/50 in...
0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,2738.492584,686.183295,Goon Edging,2052.309289,Short — Baseline redistribution — P1: 50/50 in...
0x07fd993f0fa3a185f7207adccd29f7a87404689d,160.265800,-647.664923,[ Systemic Strategies ] L/S Grids,807.930724,Short — Baseline redistribution — P1: 50/50 in...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,18730.843943,24196.229340,Realist Capital,-5465.385398,Short — Baseline redistribution — P1: 50/50 in...
0x4dec0a851849056e259128464ef28ce78afa27f6,1977.338764,6623.691469,pmalt,-4646.352705,Short — Baseline redistribution — P1: 50/50 in...
0xff9152cce6fbd30988b1aef70df6f086c99e2c55,-5039.086327,-2804.672708,gaspedal,-2234.413618,Short — Baseline redistribution — P1: 50/50 in...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,11042.826211,1561.554794,Realist Capital,9481.271417,Short — Dollar cap — P1: 50/50 inverse volatil...
0xcae0d1558b70b92ee9fd0acb20cb639c8c28ae69,7628.754169,-7.556763,DOEZOE,7636.310932,Short — Dollar cap — P1: 50/50 inverse volatil...
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,3233.314533,-191.661934,Hyperrr,3424.976467,Short — Dollar cap — P1: 50/50 inverse volatil...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0xff9152cce6fbd30988b1aef70df6f086c99e2c55,-4309.411434,-1013.303737,gaspedal,-3296.107697,Short — Dollar cap — P1: 50/50 inverse volatil...
0x4dec0a851849056e259128464ef28ce78afa27f6,2295.892682,4257.495237,pmalt,-1961.602555,Short — Dollar cap — P1: 50/50 inverse volatil...
0x565b1c32936bdaa526947178acc95c5b08cfacc4,36.478888,853.185278,Mad Scientists,-816.706390,Short — Dollar cap — P1: 50/50 inverse volatil...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,6007.279967,305.942864,Hyperrr,5701.337103,Short — Baseline redistribution — P2: Equal we...
0x2431edfcb662e6ff6deab113cc91878a0b53fb0f,3155.212552,686.183295,Goon Edging,2469.029258,Short — Baseline redistribution — P2: Equal we...
0x07fd993f0fa3a185f7207adccd29f7a87404689d,368.541568,-647.664923,[ Systemic Strategies ] L/S Grids,1016.206491,Short — Baseline redistribution — P2: Equal we...


...

,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0xff9152cce6fbd30988b1aef70df6f086c99e2c55,-3285.922589,-1021.289367,gaspedal,-2264.633222,Long — Dollar cap — P3: Regularised inverse vo...
0x04027058bb4aff889402ffb6e4743a96ce88149a,-483.801274,1438.128734,Road to 10 Million,-1921.930008,Long — Dollar cap — P3: Regularised inverse vo...
0xbbf7d7a9d0eaeab4115f022a6863450296112422,-1635.128965,-604.796043,Satori Quantum HF Vault,-1030.332921,Long — Dollar cap — P3: Regularised inverse vo...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0x3005fade4c0df5e1cd187d7062da359416f0eb8e,26376.086407,24920.079018,Delta_01,1456.007389,Long — Baseline redistribution — P4: Correlati...
0xa7f152a5f79bb5483c079610203d8fc03fd77c8e,1584.866918,835.307649,Winwin,749.559269,Long — Baseline redistribution — P4: Correlati...
0xca230e816bdb34a46960c2f978a30a563d1ae9e0,9566.811242,8859.533105,Hyperrr,707.278137,Long — Baseline redistribution — P4: Correlati...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0x4dec0a851849056e259128464ef28ce78afa27f6,3466.194159,7808.704746,pmalt,-4342.510588,Long — Baseline redistribution — P4: Correlati...
0x9e02aca9865e1859bb7865f6f64801e804a173df,-1447.489271,1716.419367,AceVault Hyper01,-3163.908639,Long — Baseline redistribution — P4: Correlati...
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,22437.891762,25183.629423,Realist Capital,-2745.737661,Long — Baseline redistribution — P4: Correlati...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0x77fee2df7bad4f1db93052fa82bf78eaab771a16,11796.089377,1573.879561,Realist Capital,10222.209816,Long — Dollar cap — P4: Correlation-aware risk...
0xda51323fe9800c8365646ad5c7ade0dd17fdc167,5363.287808,-1827.767043,Citadel,7191.054851,Long — Dollar cap — P4: Correlation-aware risk...
0xdfc24b077bc1425ad1dea75bcb6f8158e10df303,9869.463233,3923.201071,Hyperliquidity Provider (HLP),5946.262161,Long — Dollar cap — P4: Correlation-aware risk...


,Alternative PnL (USD),Control PnL (USD),Vault,Alternative minus control (USD),Run
0x82eba5dc675279cb5967952f0c4b5184505eb17c,-3301.102592,-240.721121,JizzJazz,-3060.381471,Long — Dollar cap — P4: Correlation-aware risk...
0x04027058bb4aff889402ffb6e4743a96ce88149a,-602.274758,1438.128734,Road to 10 Million,-2040.403492,Long — Dollar cap — P4: Correlation-aware risk...
0xff9152cce6fbd30988b1aef70df6f086c99e2c55,-3017.258467,-1021.289367,gaspedal,-1995.969100,Long — Dollar cap — P4: Correlation-aware risk...
